# P4: Trust check and triage (G03 Brain MRI Triage)

**Pipeline:** slice → mask (ground truth or P2 U-Net) → crop (P2) → ResNet-50 (P3) → class + confidence → Grad-CAM → IoU(heatmap, mask) → triage tier.

**Triage rule (proposal):**
- **High Priority**: glioma or meningioma, confidence > 0.80, IoU > 0.50
- **Needs Review**: confidence < 0.60 or IoU < 0.30
- **Routine**: everything else
- Added: if the prediction is *notumor*, there is no tumor region, so IoU is undefined → confidence test only.

**Fair test:** all tuning on dev. Test is run once, at the end (`RUN_TEST`).

## 1. Settings
Until P2/P3 are merged into `main`, `REPO_ROOT/P2` must come from branch `p2-tumor-cropping` and `REPO_ROOT/P3` from `p3-crop-classifier`.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/integration_project'
    REPO_ROOT = '/content/G03-brain-mri'
else:
    DATA_ROOT = os.environ.get('DATA_ROOT', r'C:/Users/DELL/Downloads/integration_project/integration_project')
    REPO_ROOT = os.environ.get('REPO_ROOT', r'C:/Users/DELL/Downloads/G03-brain-mri-main/G03-brain-mri-main')

CLASSIFIER_CKPT = os.environ.get('CLASSIFIER_CKPT', os.path.join(REPO_ROOT, 'P3/results/crop/best.pt'))  # from Mihir
UNET_CKPT       = os.environ.get('UNET_CKPT', os.path.join(REPO_ROOT, 'P2/best_unet.pth'))            # from Anika
MASK_SOURCE = os.environ.get('MASK_SOURCE', 'gt')   # 'gt' (stand-in) or 'unet' (real pipeline)
INPUT_MODE  = 'crop'                                 # 'crop' (proposal) or 'full' (baseline model)
DEV_LIMIT   = None                                   # e.g. 50 for a quick run
RUN_TEST    = False                                  # True only for the final report
OUT_DIR = os.path.join(REPO_ROOT, 'P4', 'results')
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(REPO_ROOT, 'P4'))
import trust_triage as tt
tt.setup_paths(DATA_ROOT, REPO_ROOT)
print('mask source:', MASK_SOURCE, '| input:', INPUT_MODE, '| device:', tt.pick_device())

## 2. Build the pipeline

In [ ]:
import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt

ckpt = CLASSIFIER_CKPT if os.path.exists(CLASSIFIER_CKPT) else None
if ckpt is None:
    print('WARNING: classifier checkpoint not found -> untrained model, results are meaningless (code test only)')
pipe = tt.TriagePipeline(ckpt, mask_source=MASK_SOURCE, unet_ckpt=UNET_CKPT, input_mode=INPUT_MODE)

## 3. Run on the dev split

In [ ]:
dev = pipe.run_split('dev', DATA_ROOT, limit=DEV_LIMIT)
dev.to_csv(os.path.join(OUT_DIR, f'dev_results_{MASK_SOURCE}.csv'), index=False)
tiers, by_class, iou, key = tt.summarize(dev)
print('Tier distribution'); display(tiers)
print('Tier by true class'); display(by_class)
print('Heatmap-mask IoU by true class'); display(iou)
print('Confusion matrix (rows true, cols predicted)'); display(tt.confusion(dev))
pd.Series(key)

## 4. Three complete input → output examples
One example per tier, taken from dev.

In [ ]:
from brain_mri_data import BrainMRIDataset
dev_ds = BrainMRIDataset('dev', root=DATA_ROOT)
picks = [dev.index[dev.tier == t][0] for t in tt.TIERS if (dev.tier == t).any()]
fig, axes = plt.subplots(len(picks), 4, figsize=(13, 3.4 * len(picks)), squeeze=False)
for row, i in zip(axes, picks):
    tt.plot_case(pipe.run_one(dev_ds[i], idx=i), row)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f'examples_{MASK_SOURCE}.png'), dpi=120); plt.show()

## 5. Diagnostic cases: confident mistakes
Wrong predictions with confidence > 0.8. Did the trust check (IoU) catch them?

In [ ]:
bad = dev[(dev.true != dev.pred) & (dev.confidence > 0.8)]
display(bad[['path', 'true', 'pred', 'confidence', 'iou', 'tier']].round(3))
show = list(bad.index[:4])
if show:
    fig, axes = plt.subplots(len(show), 4, figsize=(13, 3.4 * len(show)), squeeze=False)
    for row, i in zip(axes, show):
        tt.plot_case(pipe.run_one(dev_ds[i], idx=i), row)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f'confident_errors_{MASK_SOURCE}.png'), dpi=120); plt.show()

## 6. Ground-truth vs U-Net masks
When `MASK_SOURCE = 'unet'`, `iou` uses the U-Net mask and `iou_vs_gt` uses the true mask. A big gap means mask errors, not model errors, are sending cases to review.

In [ ]:
tumor = dev[dev.pred != 'notumor']
print(f"mean IoU (pipeline mask): {tumor.iou.mean():.3f} | mean IoU vs true mask: {tumor.iou_vs_gt.mean():.3f}")
moved = (tumor.iou < 0.30) & (tumor.iou_vs_gt >= 0.30)
print(f"cases sent to review only because of the mask: {int(moved.sum())} of {len(tumor)}")

## 7. Final test run (once)

In [ ]:
if RUN_TEST:
    test = pipe.run_split('test', DATA_ROOT)
    test.to_csv(os.path.join(OUT_DIR, f'test_results_{MASK_SOURCE}.csv'), index=False)
    t_tiers, t_by_class, t_iou, t_key = tt.summarize(test)
    display(t_tiers); display(t_by_class); display(t_iou); display(tt.confusion(test)); display(pd.Series(t_key))
else:
    print('Test set not used (RUN_TEST = False).')